# Transformer Models for NLP

Transformer-based NLP experiments covering DistilBERT and BERT sentiment classification, model comparison, T5 text summarization, ROUGE evaluation, and attention visualization.

> Portfolio-ready copy of the original completed experiment. Experimental code and saved outputs are preserved; assignment-administration text has been removed.


In [1]:
#@title Clean setup (run once per new Colab runtime)
!pip -q uninstall -y cudf-cu12 pylibcudf-cu12 dask-cudf-cu12 rmm-cu12 cugraph-cu12 cuml-cu12 cupy-cuda12x libcudf-cu12 || true

!pip -q install -U pip setuptools wheel
!pip -q install -U "pyarrow>=22" "datasets>=2.19.0" "transformers>=4.44.0" "accelerate>=0.33.0" "evaluate" "sentencepiece" "rouge-score" "sacrebleu"

import torch, datasets, transformers, pyarrow, evaluate
print('CUDA available:', torch.cuda.is_available())
print('pyarrow:', pyarrow.__version__)
print('datasets:', datasets.__version__)
print('transformers:', transformers.__version__)
print('evaluate:', evaluate.__version__)

CUDA available: True
pyarrow: 22.0.0
datasets: 4.4.1
transformers: 4.57.1
evaluate: 0.4.6


In [2]:
# Version-agnostic TrainingArguments helper
from packaging import version
import transformers

def make_training_args(output_dir='./results', lr=2e-5, train_bs=16, eval_bs=16, epochs=2, eval_steps=500):
    base_args = {
        'output_dir': output_dir,
        'learning_rate': lr,
        'per_device_train_batch_size': train_bs,
        'per_device_eval_batch_size': eval_bs,
        'num_train_epochs': epochs,
        'weight_decay': 0.01,
        'save_strategy': 'epoch',
        'load_best_model_at_end': True,
    }

    # Handle evaluation strategy based on version
    if version.parse(transformers.__version__) >= version.parse("4.44.0"):
        base_args['eval_strategy'] = 'epoch'
    else:
        base_args['evaluation_strategy'] = 'epoch'

    return TrainingArguments(**base_args)

Part 1 — Encoder Experiment: BERT/DistilBERT for Sentiment


In [3]:
# Import libraries
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments

# Load dataset
dataset_name = "imdb"
raw = load_dataset(dataset_name)
print("Dataset structure:", raw)
print("\nSample training example:")
print("Text:", raw['train'][0]['text'][:200] + "...")
print("Label:", raw['train'][0]['label'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset structure: DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

Sample training example:
Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...
Label: 0


In [4]:
# Initialize tokenizer and preprocessing
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def preprocess_function(batch):
    return tokenizer(batch['text'], truncation=True, max_length=256, padding=True)

tokenized = raw.map(preprocess_function, batched=True)
tokenized = tokenized.rename_column('label', 'labels')
print("Tokenized dataset:", tokenized)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Tokenized dataset: DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})


In [5]:
# Create DataCollator and model
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)

print("Model loaded successfully!")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!
Number of parameters: 66,955,010


In [6]:
# Define compute_metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [7]:
# Configure TrainingArguments with version-compatible approach
training_args = TrainingArguments(
    output_dir="./sentiment_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",  # Updated parameter name
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
)

# Use smaller subsets for faster training
train_dataset = tokenized["train"].select(range(5000))  # Reduced for speed
eval_dataset = tokenized["test"].select(range(2000))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer configured successfully!")

/tmp/ipython-input-2223684740.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Trainer configured successfully!


In [8]:
# Train and evaluate
print("Starting training...")
train_result = trainer.train()
print("Training completed!")

print("Starting evaluation...")
eval_metrics = trainer.evaluate()
print("Evaluation metrics:", eval_metrics)

Starting training...


wandb: Currently logged in as: nih00002 (nih00002-west-virginia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,0.000300,0.000176,1.000000
2,0.000100,0.000096,1.000000


Training completed!
Starting evaluation...


Evaluation metrics: {'eval_loss': 9.58511809585616e-05, 'eval_accuracy': 1.0, 'eval_runtime': 13.1618, 'eval_samples_per_second': 151.955, 'eval_steps_per_second': 9.497, 'epoch': 2.0}


Part 2 — Model Exploration & Comparison


In [9]:
# Repeat with BERT-base for comparison
model_ckpt_bert = "bert-base-uncased"

# Load tokenizer and model
tokenizer_bert = AutoTokenizer.from_pretrained(model_ckpt_bert)
model_bert = AutoModelForSequenceClassification.from_pretrained(model_ckpt_bert, num_labels=2)

# Preprocess data with BERT tokenizer
def preprocess_function_bert(batch):
    return tokenizer_bert(batch['text'], truncation=True, max_length=256, padding=True)

tokenized_bert = raw.map(preprocess_function_bert, batched=True)
tokenized_bert = tokenized_bert.rename_column('label', 'labels')

# Data collator
data_collator_bert = DataCollatorWithPadding(tokenizer=tokenizer_bert)

# Training setup
training_args_bert = TrainingArguments(
    output_dir="./bert_sentiment_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",  # Updated parameter name
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
)

train_dataset_bert = tokenized_bert["train"].select(range(5000))
eval_dataset_bert = tokenized_bert["test"].select(range(2000))

trainer_bert = Trainer(
    model=model_bert,
    args=training_args_bert,
    train_dataset=train_dataset_bert,
    eval_dataset=eval_dataset_bert,
    tokenizer=tokenizer_bert,
    data_collator=data_collator_bert,
    compute_metrics=compute_metrics,
)

# Train and evaluate BERT
print("Training BERT-base...")
train_result_bert = trainer_bert.train()
eval_metrics_bert = trainer_bert.evaluate()
print("BERT Evaluation metrics:", eval_metrics_bert)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

/tmp/ipython-input-996332493.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_bert = Trainer(


Training BERT-base...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.000200,0.000113,1.000000
2,0.000100,0.000071,1.000000


BERT Evaluation metrics: {'eval_loss': 7.110293518053368e-05, 'eval_accuracy': 1.0, 'eval_runtime': 26.1072, 'eval_samples_per_second': 76.607, 'eval_steps_per_second': 4.788, 'epoch': 2.0}


In [10]:
# Parameter count comparison
def count_params(model):
    return sum(p.numel() for p in model.parameters())

distilbert_params = count_params(model)
bert_params = count_params(model_bert)

print(f"Parameter Count Comparison:")
print(f"DistilBERT: {distilbert_params:,} parameters")
print(f"BERT-base:  {bert_params:,} parameters")
print(f"Ratio: {bert_params/distilbert_params:.2f}x more parameters in BERT")

# Performance comparison
print(f"\nPerformance Comparison:")
print(f"DistilBERT - Accuracy: {eval_metrics['eval_accuracy']:.4f}")
print(f"BERT-base  - Accuracy: {eval_metrics_bert['eval_accuracy']:.4f}")

# Runtime comparison (approximate)
print(f"\nTraining Time Comparison:")
print(f"DistilBERT: ~{train_result.metrics['train_runtime']:.1f} seconds")
print(f"BERT-base:  ~{train_result_bert.metrics['train_runtime']:.1f} seconds")

Parameter Count Comparison:
DistilBERT: 66,955,010 parameters
BERT-base:  109,483,778 parameters
Ratio: 1.64x more parameters in BERT

Performance Comparison:
DistilBERT - Accuracy: 1.0000
BERT-base  - Accuracy: 1.0000

Training Time Comparison:
DistilBERT: ~235.6 seconds
BERT-base:  ~478.9 seconds


Part 3 — Decoder/Text-to-Text Experiment: T5/FLAN-T5


In [11]:
# Import libraries for Part 3
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Trainer, TrainingArguments, Seq2SeqTrainingArguments

# Load dataset for summarization
ds = load_dataset("cnn_dailymail", "3.0.0")
print("Dataset loaded:", ds)

# Take smaller subsets for memory efficiency
train_subset = ds["train"].select(range(1000))  # Further reduced for stability
val_subset = ds["validation"].select(range(200))

Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [12]:
# Initialize tokenizer & model
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Memory-saving toggles
model.gradient_checkpointing_enable()
model.config.use_cache = False

print("T5 model loaded successfully!")

T5 model loaded successfully!


In [13]:
# Preprocess function for T5
MAX_INPUT, MAX_TARGET = 384, 96

def t5_preprocess(batch):
    # Add prefix for summarization task
    inputs = ["summarize: " + article for article in batch["article"]]

    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )

    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["highlights"],
            max_length=MAX_TARGET,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply preprocessing
processed_train = train_subset.map(t5_preprocess, batched=True, batch_size=8)
processed_val = val_subset.map(t5_preprocess, batched=True, batch_size=8)

print("Preprocessing completed!")
print("Sample processed example keys:", list(processed_train[0].keys()))

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Preprocessing completed!
Sample processed example keys: ['article', 'highlights', 'id', 'input_ids', 'attention_mask', 'labels']


In [14]:
# Data collator and metrics
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

rouge = evaluate.load("rouge")

def t5_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores - with error handling
    try:
        result = rouge.compute(
            predictions=decoded_preds,
            references=decoded_labels,
            use_stemmer=True,
            use_aggregator=True
        )

        # Extract the main ROUGE scores
        rouge_scores = {
            'rouge1': result['rouge1'] * 100,
            'rouge2': result['rouge2'] * 100,
            'rougeL': result['rougeL'] * 100,
            'rougeLsum': result['rougeLsum'] * 100,
        }

        return rouge_scores

    except Exception as e:
        print(f"Error computing ROUGE: {e}")
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0, 'rougeLsum': 0.0}

In [15]:
# TrainingArguments with OOM-safe settings
args = Seq2SeqTrainingArguments(
    output_dir="./t5_summarization",
    eval_strategy="epoch",  # Use this for newer versions
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=1,  # Reduced for demonstration
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),  # Enable if CUDA available
    logging_steps=50,
    # Remove the duplicate evaluation_strategy parameter
    save_strategy="epoch",  # Add save strategy to match eval strategy
    load_best_model_at_end=True,  # Useful for saving the best model
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=processed_train,
    eval_dataset=processed_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=t5_metrics,
)

print("T5 Trainer configured successfully!")

/tmp/ipython-input-1191319156.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


T5 Trainer configured successfully!


In [ ]:
# Train and evaluate T5 with detailed progress
print("Starting T5 training...")

# Train with progress tracking
train_result = trainer.train()

print("Training completed!")
print(f"Final training metrics: {train_result.metrics}")

# Save the final model
trainer.save_model("./t5_summarization_final")

print("Starting comprehensive evaluation...")
eval_results = trainer.evaluate()

print("\n" + "="*50)
print("T5 SUMMARIZATION EVALUATION RESULTS")
print("="*50)
print(f"Evaluation Loss: {eval_results.get('eval_loss', 'N/A'):.4f}")

# Display ROUGE scores if available
rouge_metrics = ['eval_rouge1', 'eval_rouge2', 'eval_rougeL', 'eval_rougeLsum']
for metric in rouge_metrics:
    if metric in eval_results:
        print(f"{metric}: {eval_results[metric]:.2f}")

print("="*50)

# Test the model with a few examples
print("\nSAMPLE INFERENCE TEST:")
print("="*30)

# Reset cache for inference
model.config.use_cache = True

# Test with a few examples from validation set
for i in range(min(2, len(processed_val))):
    if 'article' in processed_val[i]:
        article = processed_val[i]['article'][:300]  # First 300 chars
        inputs = tokenizer("summarize: " + article, return_tensors="pt", max_length=128, truncation=True)

        with torch.no_grad():
            summary_ids = model.generate(
                inputs.input_ids,
                max_length=60,
                num_beams=2,
                early_stopping=True,
                repetition_penalty=2.0  # Reduce repetition
            )

        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        print(f"\nExample {i+1}:")
        print(f"Input: {article[:100]}...")
        print(f"Summary: {summary}")
        print("-" * 40)

print("Model training and evaluation completed successfully!")

Starting T5 training...


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss


Part 4 — Encoder vs. Decoder Analysis


In [1]:
# Comprehensive analysis
def count_params(m):
    return sum(p.numel() for p in m.parameters())

print("=== MODEL COMPARISON ANALYSIS ===")
print(f"\nParameter Counts:")
print(f"DistilBERT: {count_params(model):,}")
print(f"BERT-base:  {count_params(model_bert):,}")
print(f"T5-small:   {count_params(model):,}")  # This shows T5 parameters correctly

# Test inference latency
import time

print(f"\n=== INFERENCE LATENCY COMPARISON ===")

# Test encoder inference
texts = ["This movie was absolutely fantastic and I loved every minute of it!",
         "I hated this film from beginning to end."]

# DistilBERT latency
print("Testing DistilBERT inference...")
tokenized_input = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=256)
start_time = time.time()
with torch.no_grad():
    outputs = model(**tokenized_input)
distilbert_time = time.time() - start_time

# BERT-base latency for comparison
print("Testing BERT-base inference...")
tokenized_input_bert = tokenizer_bert(texts, return_tensors="pt", padding=True, truncation=True, max_length=256)
start_time = time.time()
with torch.no_grad():
    outputs_bert = model_bert(**tokenized_input_bert)
bert_time = time.time() - start_time

# T5 latency (summarization) - make sure we're using the T5 model
print("Testing T5 inference...")
t5_inputs = ["summarize: " + text for text in texts]
tokenized_t5 = tokenizer(t5_inputs, return_tensors="pt", padding=True, truncation=True, max_length=128)
start_time = time.time()
with torch.no_grad():
    t5_outputs = model.generate(**tokenized_t5, max_length=50, num_beams=1, do_sample=False)
t5_time = time.time() - start_time

print(f"\nLatency Results (2 samples):")
print(f"DistilBERT (classification): {distilbert_time:.4f}s")
print(f"BERT-base  (classification): {bert_time:.4f}s")
print(f"T5-small   (generation):     {t5_time:.4f}s")
print(f"T5 is {t5_time/distilbert_time:.1f}x slower than DistilBERT")
print(f"BERT-base is {bert_time/distilbert_time:.1f}x slower than DistilBERT")

# Additional performance metrics
print(f"\n=== ADDITIONAL PERFORMANCE METRICS ===")
if 'eval_accuracy' in eval_metrics:
    print(f"DistilBERT Accuracy: {eval_metrics['eval_accuracy']:.4f}")
if 'eval_accuracy' in eval_metrics_bert:
    print(f"BERT-base Accuracy:  {eval_metrics_bert['eval_accuracy']:.4f}")

# Memory usage comparison
print(f"\n=== MEMORY CHARACTERISTICS ===")
print(f"DistilBERT: ~66M parameters (distilled version)")
print(f"BERT-base:  ~110M parameters (full version)")
print(f"T5-small:   ~60M parameters (encoder-decoder)")

print(f"\n=== FAILURE MODES ANALYSIS ===")
print("Encoder Models (BERT/DistilBERT):")
print("• Misclassification on sarcastic/ambiguous text")
print("• Poor performance on out-of-domain data")
print("• Limited context understanding beyond training")
print("• Mitigation: Data augmentation, ensemble methods, confidence calibration")

print("\nDecoder Models (T5):")
print("• Hallucination of facts not in source")
print("• Repetition in generated text")
print("• Incomplete or truncated summaries")
print("• Factual inconsistencies in generated content")
print("• Mitigation: Constrained decoding, fact verification, temperature tuning")

print(f"\n=== USE CASE RECOMMENDATIONS ===")
print("DistilBERT: Fast inference, good for real-time classification")
print("BERT-base:  Higher accuracy, better for critical classification tasks")
print("T5:         Best for text generation, summarization, translation tasks")

=== MODEL COMPARISON ANALYSIS ===

Parameter Counts:


NameError: name 'model' is not defined

Bonus — Attention Visualization


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Attention visualization for DistilBERT
def visualize_attention(text, model, tokenizer, layer=0, head=0):
    # Set model to output attentions
    original_output_attentions = model.config.output_attentions
    model.config.output_attentions = True

    try:
        # Tokenize input
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=50)

        # Get model outputs with attentions
        with torch.no_grad():
            outputs = model(**inputs, output_attentions=True)

        # Check if we have attentions
        if not outputs.attentions:
            print("No attention outputs available. The model might not support attention visualization.")
            return

        # Get attention weights for specified layer and head
        attention_weights = outputs.attentions[layer]
        attention = attention_weights[0, head].cpu().numpy()

        # Get tokens for labels
        tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

        # Remove special tokens for cleaner visualization (optional)
        clean_tokens = []
        clean_attention = []
        for i, token in enumerate(tokens):
            if token not in ['[CLS]', '[SEP]', '[PAD]']:
                clean_tokens.append(token)
                clean_attention.append(attention[i])

        # Use either original or cleaned tokens
        display_tokens = clean_tokens if clean_tokens else tokens
        display_attention = np.array(clean_attention)[:, :len(display_tokens)] if clean_tokens else attention[:len(tokens), :len(tokens)]

        # Create the plot
        plt.figure(figsize=(12, 10))
        ax = sns.heatmap(display_attention,
                        xticklabels=display_tokens,
                        yticklabels=display_tokens,
                        cmap="Blues",
                        square=True,
                        cbar_kws={'label': 'Attention Weight'})

        plt.title(f"Attention Map - Layer {layer}, Head {head}\nText: '{text}'", fontsize=14, pad=20)
        plt.xlabel("Key Tokens", fontsize=12)
        plt.ylabel("Query Tokens", fontsize=12)
        plt.xticks(rotation=45, ha='right', fontsize=10)
        plt.yticks(rotation=0, fontsize=10)
        plt.tight_layout()
        plt.show()

        # Print some statistics
        print(f"Attention statistics for Layer {layer}, Head {head}:")
        print(f"  Max attention: {attention.max():.4f}")
        print(f"  Min attention: {attention.min():.4f}")
        print(f"  Mean attention: {attention.mean():.4f}")

    except Exception as e:
        print(f"Error visualizing attention: {e}")

    finally:
        # Reset config
        model.config.output_attentions = original_output_attentions

# Test with sample text
sample_text = "The movie was fantastic and I loved the acting"
print(f"Visualizing attention for: '{sample_text}'")
visualize_attention(sample_text, model, tokenizer)

# Test with different layers and heads
print("\n" + "="*50)
print("Testing different layers and heads:")
print("="*50)

# Test a few combinations
test_combinations = [
    (0, 0),  # Layer 0, Head 0
    (2, 3),  # Layer 2, Head 3
    (5, 1),  # Layer 5, Head 1
]

for layer, head in test_combinations:
    print(f"\nLayer {layer}, Head {head}:")
    visualize_attention(sample_text, model, tokenizer, layer=layer, head=head)